# Этап 1: frozen RuBERT как признак значимости и направления

Цель — проверить, добавляет ли `mxlcw/rubert-tiny2-russian-financial-sentiment` полезный текстовый сигнал к доступным на момент решения признакам. Основная цель — `material = |abnormal return 4h| ≥ 0,5%`; направление проверяется как вторичная диагностика. Модель зафиксирована на commit `a02913e44597582218db7821d52dc15c331bf427` и не дообучается.

## План и контроль качества

1. Воспроизвести семь последовательных test-folds марта–сентября 2026 года, два предшествующих месяца validation и embargo 72 часа.
2. Не допустить пересечения `event_group_id` между train, validation и test; все labels train/validation должны быть доступны до cutoff.
3. Один раз получить frozen probabilities `positive / neutral / negative` из исходного текста новости.
4. Сравнить на одинаковых строках baseline Logistic и ту же Logistic с семью RuBERT-признаками. Для значимости primary metric — PR-AUC; для направления — hit rate, balanced accuracy и ROC-AUC.
5. Рассчитать парный cluster bootstrap по `event_group_id`, проверить стабильность по месяцам и дважды воспроизвести одинаковые артефакты.

Защитный контракт кода запрещает использовать `return_4h`, targets, `entry_at` и `label_available_at` как признаки. Точных доходностей первых пяти минут в JSONL нет, поэтому здесь измеряется добавочная ценность RuBERT к воспроизводимому baseline, а не воспроизводится заявленный коллегой ROC-AUC 0,705.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
experiment_dir = cwd if (cwd / 'run_experiment.py').exists() else cwd / 'experiments' / 'finbert_stage1'
repo_root = experiment_dir.parents[1]
sys.path.insert(0, str(repo_root))
from experiments.finbert_stage1.run_experiment import MODEL_ID, MODEL_REVISION, run_experiment
from experiments.paths import dataset_path_from_environment, stage_artifact_directory

dataset_path = dataset_path_from_environment()
artifact_dir = stage_artifact_directory('finbert_stage1')
print(f'Dataset: {dataset_path}')
print(f'Model: {MODEL_ID}@{MODEL_REVISION}')

In [ ]:
result = run_experiment(dataset_path, artifact_dir)
print('Эксперимент завершён:', result['created_at'])

In [ ]:
material = pd.read_csv(artifact_dir / 'materiality_predictions.csv')
direction = pd.read_csv(artifact_dir / 'direction_predictions.csv')
fold_metrics = pd.read_csv(artifact_dir / 'fold_metrics.csv')

assert len(material) == 480 and material['id'].nunique() == 480
assert len(direction) == 157 and direction['id'].nunique() == 157
assert set(direction['id']) == set(material.loc[material.rule_direction.isin(['up', 'down']), 'id'])
assert material['fold'].nunique() == 7
assert result['dataset']['sha256'] == 'ac639e944a100c83b91275bbf627d3e9af9c654e5a0ce726dc484c0e2c0b81b6'
print('QA passed: 1 238 source rows; 480 unique test rows; 157 paired config-6 rows; 7 folds; dataset SHA-256 fixed.')

## Значимость: итог на 480 out-of-time событиях

In [ ]:
material_rows = []
for key in ['baseline', 'with_finbert', 'finbert_direct']:
    m = result['materiality'][key]
    material_rows.append({'model': key, 'PR-AUC': m['pr_auc'], 'ROC-AUC': m['roc_auc'], 'Brier': m['brier'], 'F1': m['f1']})
display(pd.DataFrame(material_rows).set_index('model').round(4))
display(pd.DataFrame(result['materiality']['pr_auc_delta_by_fold']).round(4))
print('Paired delta:', json.dumps(result['materiality']['paired_delta_with_finbert'], ensure_ascii=False, indent=2))

## Направление: парное сравнение на 157 строках config 6

In [ ]:
direction_rows = []
for key in ['config_6', 'finbert_direct', 'logistic_baseline', 'logistic_with_finbert']:
    m = result['abnormal_direction_config6_population'][key]
    direction_rows.append({'model': key, 'hit rate': m['accuracy'], 'balanced accuracy': m['balanced_accuracy'], 'ROC-AUC': m['roc_auc'], 'raw hit rate': m['raw_direction_hit_rate']})
display(pd.DataFrame(direction_rows).set_index('model').round(4))
display(pd.DataFrame([result['abnormal_direction_config6_population']['config_6_finbert_agreement']]).round(4))
print('FinBERT, все 480 строк:', json.dumps(result['finbert_direction_full_population'], ensure_ascii=False, indent=2))

In [ ]:
display(Image(filename=str(artifact_dir / 'materiality_diagnostics.png')))

## Вывод

В текущем виде RuBERT не проходит критерий внедрения. Для значимости добавление его вероятностей подняло PR-AUC лишь с 0,6419 до 0,6438 (Δ 0,0019; cluster-bootstrap 95% CI: −0,0161…0,0206) и помогло только в 4 из 7 месяцев. Прямой sentiment-score сам по себе не связан со значимостью: ROC-AUC 0,4879.

Направление также не улучшилось: на 157 одинаковых строках config 6 и прямой RuBERT дали одинаковый hit rate 50,96%; 95% CI разницы — от −5,77 до +5,77 п.п. Согласие двух методов покрывает 86,0% строк, но даёт только 51,11% попаданий. Следующий рациональный шаг — не внедрение этой модели, а получение отсутствующих exact 0–5m market features и затем тест текстовых embedding-признаков или supervised fine-tuning на materiality.